In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

for rel_path in ['../', '../../', '../../../']:
    abs_path = os.path.abspath(rel_path)
    if abs_path not in sys.path:
        sys.path.insert(0, abs_path)

from benchmarking.utils import read_perf_eval_json_files

COLOR_MAP    = {'Encrypted': '#EF553B', 'Non-Encrypted': '#636EFA'}
SYMBOL_MAP   = {'Encrypted': 'circle',  'Non-Encrypted': 'square'}
CONC_ORDER   = [1, 2, 4, 8, 16, 32]
CONC_STR     = [str(c) for c in CONC_ORDER]

print('Setup complete')

Setup complete


# DataKrypto TEE Performance Analysis
## Encrypted vs Non-Encrypted Model — Benchmarking Report

---

### Architecture

```
Non-Encrypted:
  Client ──────────────────────────► SambaNova Inference Server
                    (1 network hop)

Encrypted:
  Client ──► DataKrypto TEE ───────► SambaNova Inference Server
              │  encrypts input          returns encrypted chunks
              │  decrypts each output ◄─────────────────────────┘
              │  chunk (streaming)
              └──► Client
              (2 network hops total)
```

### Models
| Model | Description |
|-------|-------------|
| `Llama-xLAM-2-8b-fc-r` | Base model (plain inference) |
| `Llama-xLAM-2-8b-fc-r-encrypted` | Runs through DataKrypto TEE |

> Note: The two models are served by **separate SambaNova endpoints** which may have
> different hardware allocations or decode configurations. Any baseline performance
> difference between them is an endpoint-level characteristic — not caused by the TEE.

### Test Matrix
- **Input tokens**: 2,000 (fixed)
- **Output tokens**: 100 and 1,024
- **Concurrencies**: 1, 2, 4, 8, 16, 32
- Each concurrency level = **N separate HTTP requests sent simultaneously** (not a single batch API call — N independent requests hitting the server at the same time, which the server can dynamically batch together for inference)
- **Single Run (1x)**: One round of N simultaneous requests — clean first-hit baseline at each concurrency level
- **Multi Run (10x)**: 10 sequential rounds of the same N simultaneous requests — same per-round server load as Single Run, but repeated 10 times. Provides statistical robustness and tests sustained server/TEE behavior over time

> Key difference: Single and Multi Run have the **same per-round concurrency load**.
> Multi Run adds statistical reliability (more samples) and reveals sustained-load behavior —
> it does not increase the number of simultaneous requests per round.

### TEE Overhead Sources
1. **Input Encryption** — encrypts the prompt (~4–10 ms, near-constant per request)
2. **Output Decryption** — decrypts each streaming token chunk (grows with output tokens)
3. **Network Hop** — extra Client → TEE → Server roundtrip (dominant overhead)

> **TEE capacity constraint**: The TEE has **8 cores** and processes encryption/decryption
> sequentially per core. At concurrency > 8, the TEE queue fills up causing requests to wait,
> which means concurrent requests may arrive at the inference server **staggered** rather than
> simultaneously — reducing the effective batch size the server sees.

---

In [ ]:
DATA_ROOT = os.path.abspath('../../data/datakrypto')

DATASETS = {
    'enc_simple': {
        'consolidated': f'{DATA_ROOT}/speed_bench_test_encrypted_simple/consolidated_results/20260225-155057.028751.xlsx',
        'run_dir':      f'{DATA_ROOT}/speed_bench_test_encrypted_simple/20260225-155057.028751',
        'encrypted': True,  
        'run_type': 'Single Run (1x)', 
        'label': 'Encrypted',
    },
    'plain_simple': {
        'consolidated': f'{DATA_ROOT}/speed_bench_test_non_encrypted_simple/consolidated_results/20260227-204607.930274.xlsx',
        'run_dir':      f'{DATA_ROOT}/speed_bench_test_non_encrypted_simple/20260227-204607.930274',
        'encrypted': False, 
        'run_type': 'Single Run (1x)', 
        'label': 'Non-Encrypted',
    },
    'enc_multiple': {
        'consolidated': f'{DATA_ROOT}/speed_bench_test_encrypted_multiple/consolidated_results/20260225-150147.253373.xlsx',
        'run_dir':      f'{DATA_ROOT}/speed_bench_test_encrypted_multiple/20260225-150147.253373',
        'encrypted': True,  
        'run_type': 'Multi Run (10x)', 
        'label': 'Encrypted',
    },
    'plain_multiple': {
        'consolidated': f'{DATA_ROOT}/speed_bench_test_non_encrypted_multiple/consolidated_results/20260227-205436.463636.xlsx',
        'run_dir':      f'{DATA_ROOT}/speed_bench_test_non_encrypted_multiple/20260227-205436.463636',
        'encrypted': False, 
        'run_type': 'Multi Run (10x)', 
        'label': 'Non-Encrypted',
    },
}

dfs = []
for key, meta in DATASETS.items():
    df = pd.read_excel(meta['consolidated'])
    df['dataset_key'] = key
    df['encrypted']   = meta['encrypted']
    df['run_type']    = meta['run_type']
    df['label']       = meta['label']
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

# Derived columns
df_all['out_tokens_label']    = df_all['num_output_tokens'].astype(str) + ' output tokens'
df_all['total_requests']      = df_all['num_completed_requests'] + df_all['number_errors']
df_all['error_rate_pct']      = (df_all['number_errors'] / df_all['total_requests'].replace(0, np.nan)) * 100
df_all['completion_rate_pct'] = 100 - df_all['error_rate_pct']

# Ordered categorical for consistent facet ordering
df_all['run_type'] = pd.Categorical(
    df_all['run_type'],
    categories=['Single Run (1x)', 'Multi Run (10x)'],
    ordered=True
)

print(f'Loaded {len(df_all)} rows across {df_all["dataset_key"].nunique()} datasets')
df_all.groupby(['label', 'run_type'])[['num_concurrent_requests']].count().rename(columns={'num_concurrent_requests': 'scenarios'})

## 1. Dataset Overview

In [4]:
cols = ['label', 
        'run_type', 
        'num_output_tokens', 
        'num_concurrent_requests',
        'num_completed_requests', 
        'number_errors', 
        'error_rate_pct'
        ]
overview = df_all[cols].copy()
overview.columns = ['Model', 
                    'Run Type', 
                    'Output Tokens', 
                    'Concurrency',
                    'Completed', 
                    'Errors', 
                    'Error Rate (%)'
                    ]
overview['Error Rate (%)'] = overview['Error Rate (%)'].fillna(0).round(1)
overview = overview.sort_values(['Run Type', 'Model', 'Output Tokens', 'Concurrency'])

def _highlight_errors(row):
    if row['Errors'] > 0:
        return ['background-color: #ffdddd'] * len(row)
    return [''] * len(row)

overview.style.apply(_highlight_errors, axis=1).format({'Error Rate (%)': '{:.1f}%'})

,Model,Run Type,Output Tokens,Concurrency,Completed,Errors,Error Rate (%)
0,Encrypted,Single Run (1x),100,1,1,0,0.0%
1,Encrypted,Single Run (1x),100,2,2,0,0.0%
2,Encrypted,Single Run (1x),100,4,4,0,0.0%
3,Encrypted,Single Run (1x),100,8,8,0,0.0%
4,Encrypted,Single Run (1x),100,16,16,0,0.0%
5,Encrypted,Single Run (1x),100,32,32,0,0.0%
6,Encrypted,Single Run (1x),1024,1,1,0,0.0%
7,Encrypted,Single Run (1x),1024,2,2,0,0.0%
8,Encrypted,Single Run (1x),1024,4,4,0,0.0%
9,Encrypted,Single Run (1x),1024,8,8,0,0.0%


## 2. Error & Completion Rate Analysis

The TEE has **8 cores** processing encryption/decryption sequentially.
At concurrency > 8, the TEE queue fills up and requests may time out or be rejected,
producing errors on the encrypted model only. The plain model should show zero errors.

In [5]:
fig = px.bar(
    df_all,
    x='num_concurrent_requests',
    y='error_rate_pct',
    color='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    barmode='group',
    color_discrete_map=COLOR_MAP,
    category_orders={
        'num_concurrent_requests': CONC_ORDER,
        'run_type': ['Single Run (1x)', 'Multi Run (10x)'],
    },
    labels={
        'num_concurrent_requests': 'Concurrency',
        'error_rate_pct':          'Error Rate (%)',
        'label':                   'Model',
        'out_tokens_label':        'Output Tokens',
        'run_type':                'Run Type',
    },
    title=(
        'Request Error Rate by Concurrency<br>'
        '<sub>Errors appear on Encrypted model at concurrency > 8 (TEE queue saturation). '
        'Non-Encrypted should be 0% throughout.</sub>'
    ),
)
fig.update_layout(height=600, width=1100, template='plotly_white')
fig.update_xaxes(type='category')
fig.show()

In [7]:
# Completed requests per minute — real system throughput capacity
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='num_completed_requests_per_min',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests':      'Concurrency (log scale)',
        'num_completed_requests_per_min': 'Completed Requests / min',
        'label':                         'Model',
        'out_tokens_label':              'Output Tokens',
        'run_type':                      'Run Type',
    },
    title=(
        'System Throughput: Completed Requests per Minute<br>'
        '<sub>ndicator of system capacity. '
        'Encrypted model throughput is severely limited by TEE queueing at higher concurrencies.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

## 3. Server-Side Metrics — Assumption Validation ✅

With both endpoints on equivalent configurations, server-side metrics now confirm the core assumption:
**the TEE does not affect inference speed inside the SambaNova engine.**

**Server tok/s — endpoints are now matched:**
- Both endpoints baseline at ~585–620 tok/s (ratio 1.02–1.06× at concurrency 1–2 and 8+)
- A small anomaly remains at concurrency 4 (ratio ~1.33–1.46×) which may be a transient batching artifact
- The previous 1.87× gap was an endpoint configuration issue, now resolved

**Server TTFT — two distinct regimes clearly visible:**
- **Concurrency 1–4**: Both models show nearly identical server TTFT (~0.091 s and ~0.161 s respectively)
  — direct confirmation that the TEE has no impact on server prefill latency
- **Concurrency 8–32**: Encrypted stays flat at ~0.163 s; non-encrypted jumps to ~0.311 s.
  This divergence is the **TEE de-batching effect** in action: because the TEE processes
  requests sequentially per core, they arrive at the server staggered rather than simultaneously,
  keeping the effective server batch smaller and prefill latency lower for the encrypted path.

**Interesting consequence at high concurrency:** The encrypted model's server-side advantage from
de-batching (lower TTFT) can exceed the TEE network overhead, making encrypted client TTFT
actually lower than non-encrypted at concurrency ≥ 16 for 100-token responses (see Section 5).

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='server_ttft_s_p50',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests': 'Concurrency (log scale)',
        'server_ttft_s_p50':       'Server TTFT p50 (s)',
        'label':                   'Model',
        'out_tokens_label':        'Output Tokens',
        'run_type':                'Run Type',
    },
    title=(
        'Server Time to First Token — p50<br>'
        '<sub>Concurrency 1–4: nearly identical for both models ✅ — confirms TEE has no server-side impact. '
        'Concurrency 8–32: encrypted stays flat (~0.163 s), non-encrypted rises to ~0.311 s. '
        'This split is the TEE de-batching effect: staggered arrival keeps the effective server batch '
        'smaller for the encrypted path, reducing prefill queue time.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='server_output_token_per_s_p50',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests':      'Concurrency (log scale)',
        'server_output_token_per_s_p50': 'Server Output Tok/s p50 (per request)',
        'label':                         'Model',
        'out_tokens_label':              'Output Tokens',
        'run_type':                      'Run Type',
    },
    title=(
        'Server Output Throughput per Request — p50<br>'
        '<sub>Both endpoints now matched (~585–620 tok/s). Small gap at conc=4 may be a batching artifact. '
        'Encrypted stays flat across all concurrencies (TEE de-batching keeps effective server batch small). '
        'Non-Encrypted baseline is slightly higher but both degrade similarly at large batches.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [11]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='server_end_to_end_latency_s_p50',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests':          'Concurrency (log scale)',
        'server_end_to_end_latency_s_p50':  'Server E2E Latency p50 (s)',
        'label':                            'Model',
        'out_tokens_label':                 'Output Tokens',
        'run_type':                         'Run Type',
    },
    title='Server End-to-End Latency — p50',
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [12]:
# Server metrics delta table — Encrypted minus Non-Encrypted
DIMS = ['run_type', 'num_output_tokens', 'num_concurrent_requests']

df_enc_s   = df_all[df_all['label'] == 'Encrypted'].set_index(DIMS)
df_plain_s = df_all[df_all['label'] == 'Non-Encrypted'].set_index(DIMS)

server_metrics = [
    ('server_ttft_s_p50',              'Server TTFT p50 (s)'),
    ('server_output_token_per_s_p50',  'Server Tok/s p50'),
    ('server_end_to_end_latency_s_p50','Server E2E p50 (s)'),
]

delta_rows = []
for col, label in server_metrics:
    if col not in df_enc_s.columns:
        continue
    for idx in df_enc_s.index:
        if idx not in df_plain_s.index:
            continue
        enc_val   = df_enc_s.loc[idx, col]
        plain_val = df_plain_s.loc[idx, col]
        if pd.isna(enc_val) or pd.isna(plain_val):
            continue
        delta   = enc_val - plain_val
        pct_chg = delta / plain_val * 100 if plain_val != 0 else np.nan
        delta_rows.append({
            'Metric':          label,
            'Run Type':        idx[0],
            'Output Tokens':   idx[1],
            'Concurrency':     idx[2],
            'Non-Encrypted':   plain_val,
            'Encrypted':       enc_val,
            'Delta':           delta,
            '% Change':        pct_chg,
        })

df_server_delta = pd.DataFrame(delta_rows)
df_server_delta = df_server_delta.sort_values(['Metric', 'Run Type', 'Output Tokens', 'Concurrency'])

def _pct_color(val):
    if pd.isna(val): return ''
    if val >  50: return 'background-color: #ffaaaa'
    if val >  20: return 'background-color: #ffd8aa'
    if val >   0: return 'background-color: #fffacc'
    if val < -20: return 'background-color: #aaffaa'
    return ''

df_server_delta.style \
    .applymap(_pct_color, subset=['% Change']) \
    .format({
        'Non-Encrypted': '{:.4f}',
        'Encrypted':     '{:.4f}',
        'Delta':         '{:+.4f}',
        '% Change':      '{:+.1f}%',
    })

,Metric,Run Type,Output Tokens,Concurrency,Non-Encrypted,Encrypted,Delta,% Change
58,Server E2E p50 (s),Multi Run (10x),100,1,0.2509,0.2604,+0.0095,+3.8%
59,Server E2E p50 (s),Multi Run (10x),100,2,0.2511,0.2592,+0.0081,+3.2%
60,Server E2E p50 (s),Multi Run (10x),100,4,0.2881,0.2616,-0.0265,-9.2%
61,Server E2E p50 (s),Multi Run (10x),100,8,0.4740,0.2910,-0.1830,-38.6%
62,Server E2E p50 (s),Multi Run (10x),100,16,0.4743,0.2936,-0.1807,-38.1%
63,Server E2E p50 (s),Multi Run (10x),100,32,0.4752,0.2927,-0.1825,-38.4%
64,Server E2E p50 (s),Multi Run (10x),1024,1,1.7395,1.8069,+0.0674,+3.9%
65,Server E2E p50 (s),Multi Run (10x),1024,2,1.7399,1.7854,+0.0455,+2.6%
66,Server E2E p50 (s),Multi Run (10x),1024,4,1.0369,1.5503,+0.5134,+49.5%
67,Server E2E p50 (s),Multi Run (10x),1024,8,1.4285,1.4902,+0.0617,+4.3%


## 4. TEE Overhead Decomposition (Encrypted Model Only)

Three measurable components are returned by the TEE per request:

| Component | Description | Expected Behavior |
|-----------|-------------|-------------------|
| **Input Encryption** | Time to encrypt the prompt before sending to server | ~constant, independent of output length |
| **Output Decryption** | Cumulative time to decrypt all streaming output chunks | grows with output token count |
| **Network Hop** | Round-trip latency through the TEE (Client→TEE→Server→TEE→Client) | grows with concurrency (TEE queue) and output tokens |

> The network hop is the dominant overhead. At high concurrency it grows due to TEE core saturation.

In [13]:
df_tee = df_all[df_all['label'] == 'Encrypted'].copy()

# Convert ms → s for consistent units with latency charts
df_tee['Network Hop']        = df_tee['server_network_latency_ms_mean'] / 1000
df_tee['Input Encryption']   = df_tee['total_encryption_time_ms_mean']  / 1000
df_tee['Output Decryption']  = df_tee['total_decryption_time_ms_mean']  / 1000

TEE_ID_COLS  = ['run_type', 'out_tokens_label', 'num_concurrent_requests']
TEE_VAL_COLS = ['Network Hop', 'Input Encryption', 'Output Decryption']

df_tee_melt = df_tee[TEE_ID_COLS + TEE_VAL_COLS].melt(
    id_vars=TEE_ID_COLS, var_name='Component', value_name='Overhead (s)'
)

COMP_COLORS = {
    'Network Hop':       '#FF7F0E',
    'Input Encryption':  '#D62728',
    'Output Decryption': '#2CA02C',
}

fig = px.bar(
    df_tee_melt,
    x='num_concurrent_requests',
    y='Overhead (s)',
    color='Component',
    facet_col='out_tokens_label',
    facet_row='run_type',
    barmode='stack',
    color_discrete_map=COMP_COLORS,
    category_orders={
        'num_concurrent_requests': CONC_ORDER,
        'run_type': ['Single Run (1x)', 'Multi Run (10x)'],
    },
    labels={
        'num_concurrent_requests': 'Concurrency',
        'run_type':                'Run Type',
        'out_tokens_label':        'Output Tokens',
    },
    title=(
        'TEE Overhead Breakdown per Request (Encrypted Model Only)<br>'
        '<sub>Network Hop dominates. Encryption is near-constant. '
        'Decryption scales with output token count (streaming per-chunk decryption).</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(type='category')
fig.show()

In [14]:
# Network latency growth — most important overhead component
fig = px.line(
    df_tee,
    x='num_concurrent_requests',
    y='server_network_latency_ms_mean',
    color='out_tokens_label',
    symbol='run_type',
    line_dash='run_type',
    line_dash_map={'Single Run (1x)': 'solid', 'Multi Run (10x)': 'dash'},
    facet_col='out_tokens_label',
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests':        'Concurrency (log scale)',
        'server_network_latency_ms_mean': 'TEE Network Latency — Mean (ms)',
        'out_tokens_label':               'Output Tokens',
        'run_type':                       'Run Type',
    },
    title=(
        'TEE Network Latency vs Concurrency<br>'
        '<sub>Solid = Single Run (1x) | Dashed = Multi Run (10x). '
        'Grows with concurrency (TEE queueing) AND with output tokens '
        '(streaming: each decrypted chunk adds a network round-trip).</sub>'
    ),
)
fig.update_layout(height=450, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [15]:
# Encryption and decryption times detail
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Input Encryption Time (ms)', 'Output Decryption Time (ms)'],
)

_run_styles  = {'Single Run (1x)': 'solid', 'Multi Run (10x)': 'dash'}
_tok_colors  = {100: '#1f77b4', 1024: '#ff7f0e'}
_shown       = set()

for run_type, dash in _run_styles.items():
    for out_tok, color in _tok_colors.items():
        sub  = df_tee[(df_tee['run_type'] == run_type) & (df_tee['num_output_tokens'] == out_tok)]
        name = f'{out_tok} out | {run_type}'
        show = name not in _shown
        kw   = dict(x=sub['num_concurrent_requests'], mode='lines+markers',
                    name=name, line=dict(color=color, dash=dash),
                    legendgroup=name, showlegend=show)
        fig.add_trace(go.Scatter(y=sub['total_encryption_time_ms_mean'], **kw), row=1, col=1)
        kw['showlegend'] = False
        fig.add_trace(go.Scatter(y=sub['total_decryption_time_ms_mean'], **kw), row=1, col=2)
        _shown.add(name)

fig.update_xaxes(
    type='log', tickvals=CONC_ORDER, ticktext=CONC_STR,
    title_text='Concurrency (log scale)',
)
fig.update_yaxes(title_text='Time (ms)', col=1)
fig.update_yaxes(title_text='Time (ms)', col=2)
fig.update_layout(
    height=420, width=1100, template='plotly_white',
    legend_title='out tokens | run type',
    title=(
        'TEE Encryption & Decryption Times<br>'
        '<sub>Encryption is near-constant (~4-10 ms). '
        'Decryption grows with output token count; decreases at higher concurrency '
        'as TEE queue pressure compresses per-request wall time.</sub>'
    ),
)
fig.show()

## 5. Client-Side Metrics — End-User Experience

Client metrics include the full roundtrip: **Client → (TEE) → Server → (TEE) → Client**.

The gap between encrypted and non-encrypted client metrics is the
**observable user-facing cost** of the TEE encryption layer.

In [ ]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_ttft_s_p50',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests': 'Concurrency (log scale)',
        'client_ttft_s_p50':       'Client TTFT p50 (s)',
        'label':                   'Model',
        'out_tokens_label':        'Output Tokens',
        'run_type':                'Run Type',
    },
    title=(
        'Client Time to First Token — p50<br>'
        '<sub>At concurrency 1–8: encrypted is slower due to TEE network overhead. '
        'At concurrency ≥ 16 (100 tokens): encrypted TTFT drops below non-encrypted — '
        'the TEE de-batching advantage on the server (lower prefill time) outweighs '
        'the network overhead, flipping the comparison. This crossover does not appear '
        'for 1,024 tokens where network overhead is larger and dominates.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [17]:
# Client TTFT overhead attribution: stacked TEE components vs total delta
DIMS = ['run_type', 'num_output_tokens', 'num_concurrent_requests']

df_enc_a   = df_all[df_all['label'] == 'Encrypted'].copy()
df_plain_a = df_all[df_all['label'] == 'Non-Encrypted'][DIMS + ['client_ttft_s_p50']].rename(
    columns={'client_ttft_s_p50': 'client_ttft_plain'}
)
df_attr = df_enc_a.merge(df_plain_a, on=DIMS)

df_attr['client_ttft_delta']   = df_attr['client_ttft_s_p50'] - df_attr['client_ttft_plain']
df_attr['Network Hop (s)']     = df_attr['server_network_latency_ms_mean'] / 1000
df_attr['Input Encryption (s)']= df_attr['total_encryption_time_ms_mean']  / 1000
df_attr['Output Decryption (s)']= df_attr['total_decryption_time_ms_mean'] / 1000
df_attr['Residual (s)']        = (
    df_attr['client_ttft_delta']
    - df_attr['Network Hop (s)']
    - df_attr['Input Encryption (s)']
    - df_attr['Output Decryption (s)']
).clip(lower=0)
df_attr['out_tokens_label'] = df_attr['num_output_tokens'].astype(str) + ' output tokens'

ATTR_COLS   = ['Network Hop (s)', 'Input Encryption (s)', 'Output Decryption (s)', 'Residual (s)']
ATTR_COLORS = {
    'Network Hop (s)':      '#FF7F0E',
    'Input Encryption (s)': '#D62728',
    'Output Decryption (s)':'#2CA02C',
    'Residual (s)':         '#9467BD',
}

df_attr_melt = df_attr.melt(
    id_vars=['run_type', 'out_tokens_label', 'num_concurrent_requests', 'client_ttft_delta'],
    value_vars=ATTR_COLS, var_name='Component', value_name='Overhead (s)',
).dropna(subset=['Overhead (s)'])

fig = px.bar(
    df_attr_melt,
    x='num_concurrent_requests',
    y='Overhead (s)',
    color='Component',
    facet_col='out_tokens_label',
    facet_row='run_type',
    barmode='stack',
    color_discrete_map=ATTR_COLORS,
    category_orders={
        'num_concurrent_requests': CONC_ORDER,
        'run_type': ['Single Run (1x)', 'Multi Run (10x)'],
    },
    labels={
        'num_concurrent_requests': 'Concurrency',
        'run_type':                'Run Type',
        'out_tokens_label':        'Output Tokens',
    },
    title=(
        'Client TTFT Overhead Attribution: Encrypted − Non-Encrypted Delta<br>'
        '<sub>Stacked bars = measured TEE components. Total bar height = Enc TTFT − Plain TTFT. '
        'Residual = delta not fully explained by TEE metrics (measurement noise / base model diff).</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(type='category')
fig.show()

In [18]:
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_end_to_end_latency_s_p50',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests':         'Concurrency (log scale)',
        'client_end_to_end_latency_s_p50': 'Client E2E Latency p50 (s)',
        'label':                           'Model',
        'out_tokens_label':                'Output Tokens',
        'run_type':                        'Run Type',
    },
    title=(
        'Client End-to-End Latency — p50<br>'
        '<sub>Full request duration as observed by the client. '
        'Grows faster for encrypted at high concurrency due to TEE queueing.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [19]:
# Per-request client generation speed
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_output_token_per_s_p50',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests':        'Concurrency (log scale)',
        'client_output_token_per_s_p50':  'Client Output Tok/s p50 (per request)',
        'label':                          'Model',
        'out_tokens_label':               'Output Tokens',
        'run_type':                       'Run Type',
    },
    title=(
        'Client Output Throughput per Request — p50<br>'
        '<sub>After the first token, subsequent tokens stream at this rate. '
        'Encrypted is limited by TEE streaming decryption speed; '
        'Non-Encrypted is limited by server generation speed at high concurrency.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

In [21]:
# Total system throughput — aggregate across all concurrent requests
fig = px.line(
    df_all,
    x='num_concurrent_requests',
    y='client_total_output_throughput',
    color='label',
    symbol='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    symbol_map=SYMBOL_MAP,
    log_x=True,
    markers=True,
    labels={
        'num_concurrent_requests':    'Concurrency (log scale)',
        'client_total_output_throughput': 'Total Output Throughput (tok/s)',
        'label':                      'Model',
        'out_tokens_label':           'Output Tokens',
        'run_type':                   'Run Type',
    },
    title=(
        'Total Client Output Throughput — All Concurrent Requests<br>'
        '<sub>Non-Encrypted scales nearly linearly with concurrency. '
        'Encrypted collapses at higher concurrencies due to TEE queuing adding large wall-clock delays.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.update_xaxes(tickvals=CONC_ORDER, ticktext=CONC_STR)
fig.show()

## 6. Per-Request Distributions

Box plots using individual request data from all runs.
- **Multi Run (10x)** provides the richest distributions (10–320 requests per scenario across 10 rounds).
- **Single Run (1x)** covers only the N simultaneous requests in a single round (1–32 points per box depending on concurrency).

Error requests (where `error_code` is set) are excluded.

In [22]:
def _parse_fname(fname):
    """Extract (out_tokens, concurrency) from benchmark filename."""
    base  = os.path.basename(str(fname)).replace('_individual_responses.json', '')
    parts = base.split('_')
    try:
        return int(parts[4]), int(parts[5])
    except (IndexError, ValueError):
        return None, None

dfs_ind = []
for key, meta in DATASETS.items():
    try:
        df_ind = read_perf_eval_json_files(meta['run_dir'], type='individual_responses')
        if df_ind.empty:
            continue
        df_ind['label']    = meta['label']
        df_ind['run_type'] = meta['run_type']
        parsed = df_ind['filename'].apply(lambda f: pd.Series(_parse_fname(f)))
        df_ind['num_output_tokens']       = parsed[0]
        df_ind['num_concurrent_requests'] = parsed[1]
        dfs_ind.append(df_ind)
    except Exception as e:
        print(f'Warning: could not load {key}: {e}')

df_individual = pd.concat(dfs_ind, ignore_index=True)
df_individual = df_individual.dropna(
    subset=['num_output_tokens', 'num_concurrent_requests', 'client_ttft_s']
)
df_individual['num_output_tokens']       = df_individual['num_output_tokens'].astype(int)
df_individual['num_concurrent_requests'] = df_individual['num_concurrent_requests'].astype(int)
df_individual['out_tokens_label']        = df_individual['num_output_tokens'].astype(str) + ' output tokens'
df_individual['concurrency_str']         = df_individual['num_concurrent_requests'].astype(str)
df_individual['is_error']                = df_individual['error_code'].notna()
df_individual['run_type'] = pd.Categorical(
    df_individual['run_type'],
    categories=['Single Run (1x)', 'Multi Run (10x)'],
    ordered=True
)

print(f'Loaded {len(df_individual):,} individual request records')
df_individual.groupby(['label', 'run_type', 'is_error'])[['client_ttft_s']].count().rename(
    columns={'client_ttft_s': 'count'}
)

Loaded 2,621 individual request records


count
label         run_type        is_error       
Encrypted     Single Run (1x) False       126
              Multi Run (10x) False      1109
Non-Encrypted Single Run (1x) False       126
              Multi Run (10x) False      1260

In [23]:
df_box = df_individual[~df_individual['is_error']].copy()

fig = px.box(
    df_box,
    x='concurrency_str',
    y='client_ttft_s',
    color='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    category_orders={
        'concurrency_str': CONC_STR,
        'run_type': ['Single Run (1x)', 'Multi Run (10x)'],
    },
    points='outliers',
    labels={
        'concurrency_str': 'Concurrency',
        'client_ttft_s':   'Client TTFT (s)',
        'label':           'Model',
        'out_tokens_label':'Output Tokens',
        'run_type':        'Run Type',
    },
    title=(
        'Client TTFT Distribution per Concurrency<br>'
        '<sub>Encrypted shows higher median AND wider spread due to variable TEE network latency. '
        'High concurrency increases spread (TEE queue wait time variance).</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.show()

In [24]:
fig = px.box(
    df_box,
    x='concurrency_str',
    y='server_ttft_s',
    color='label',
    facet_col='out_tokens_label',
    facet_row='run_type',
    color_discrete_map=COLOR_MAP,
    category_orders={
        'concurrency_str': CONC_STR,
        'run_type': ['Single Run (1x)', 'Multi Run (10x)'],
    },
    points='outliers',
    labels={
        'concurrency_str': 'Concurrency',
        'server_ttft_s':   'Server TTFT (s)',
        'label':           'Model',
        'out_tokens_label':'Output Tokens',
        'run_type':        'Run Type',
    },
    title=(
        'Server TTFT Distribution per Concurrency<br>'
        '<sub>Server-side distribution. At medium concurrencies, encrypted may show LOWER server TTFT '
        'because TEE de-batching reduces effective batch size seen by the server.</sub>'
    ),
)
fig.update_layout(height=700, width=1100, template='plotly_white')
fig.show()

In [25]:
# Per-request: TEE network latency vs server TTFT (encrypted multi run only)
df_enc_ind = df_individual[
    (df_individual['label']    == 'Encrypted') &
    (df_individual['run_type'] == 'Multi Run (10x)') &
    (~df_individual['is_error']) &
    (df_individual['server_network_latency_ms'].notna())
].copy()

fig = px.scatter(
    df_enc_ind,
    x='server_ttft_s',
    y='server_network_latency_ms',
    color='out_tokens_label',
    facet_col='concurrency_str',
    opacity=0.55,
    category_orders={'concurrency_str': CONC_STR},
    labels={
        'server_ttft_s':             'Server TTFT (s)',
        'server_network_latency_ms': 'TEE Network Latency (ms)',
        'out_tokens_label':          'Output Tokens',
        'concurrency_str':           'Concurrency',
    },
    title=(
        'Per-Request: TEE Network Latency vs Server TTFT — Encrypted Multi Run<br>'
        '<sub>Shows how network latency varies independently from server TTFT. '
        'A flat horizontal spread would indicate network latency is independent of server load.</sub>'
    ),
)
fig.update_layout(height=420, width=1400, template='plotly_white')
fig.show()

## 7. Summary

### Full Metric Comparison

In [26]:
DIMS = ['run_type', 'num_output_tokens', 'num_concurrent_requests']

ALL_METRICS = [
    ('server_ttft_s_p50',               'Server TTFT p50 (s)'),
    ('server_output_token_per_s_p50',   'Server Tok/s p50'),
    ('server_end_to_end_latency_s_p50', 'Server E2E p50 (s)'),
    ('client_ttft_s_p50',               'Client TTFT p50 (s)'),
    ('client_end_to_end_latency_s_p50', 'Client E2E p50 (s)'),
    ('client_output_token_per_s_p50',   'Client Tok/s p50'),
    ('client_total_output_throughput',  'Total Throughput (tok/s)'),
    ('num_completed_requests_per_min',  'Req / min'),
    ('error_rate_pct',                  'Error Rate (%)'),
]

df_enc_s   = df_all[df_all['label'] == 'Encrypted'].set_index(DIMS)
df_plain_s = df_all[df_all['label'] == 'Non-Encrypted'].set_index(DIMS)

rows = []
for col, label in ALL_METRICS:
    if col not in df_enc_s.columns:
        continue
    for idx in df_enc_s.index:
        if idx not in df_plain_s.index:
            continue
        enc_v   = df_enc_s.loc[idx, col]
        plain_v = df_plain_s.loc[idx, col]
        if pd.isna(enc_v) and pd.isna(plain_v):
            continue
        delta   = enc_v - plain_v if pd.notna(enc_v) and pd.notna(plain_v) else np.nan
        pct_chg = delta / plain_v * 100 if pd.notna(delta) and plain_v != 0 else np.nan
        rows.append({
            'Metric':        label,
            'Run Type':      idx[0],
            'Output Tokens': idx[1],
            'Concurrency':   idx[2],
            'Non-Encrypted': plain_v,
            'Encrypted':     enc_v,
            'Delta':         delta,
            '% Change':      pct_chg,
        })

df_summary = pd.DataFrame(rows).sort_values(
    ['Metric', 'Run Type', 'Output Tokens', 'Concurrency']
)

def _pct_color(val):
    if pd.isna(val): return ''
    if val >  100: return 'background-color: #ff6666'
    if val >   50: return 'background-color: #ffaaaa'
    if val >   20: return 'background-color: #ffd8aa'
    if val >    0: return 'background-color: #fffacc'
    if val < -20:  return 'background-color: #aaffaa'
    return ''

df_summary.style \
    .applymap(_pct_color, subset=['% Change']) \
    .format({
        'Non-Encrypted': '{:.3f}',
        'Encrypted':     '{:.3f}',
        'Delta':         '{:+.3f}',
        '% Change':      '{:+.1f}%',
    }, na_rep='—')

,Metric,Run Type,Output Tokens,Concurrency,Non-Encrypted,Encrypted,Delta,% Change
108,Client E2E p50 (s),Multi Run (10x),100,1,0.377,0.504,+0.127,+33.7%
109,Client E2E p50 (s),Multi Run (10x),100,2,0.634,0.658,+0.024,+3.8%
110,Client E2E p50 (s),Multi Run (10x),100,4,0.658,0.685,+0.027,+4.1%
111,Client E2E p50 (s),Multi Run (10x),100,8,0.853,0.838,-0.015,-1.7%
112,Client E2E p50 (s),Multi Run (10x),100,16,1.073,0.791,-0.282,-26.3%
113,Client E2E p50 (s),Multi Run (10x),100,32,2.165,0.918,-1.247,-57.6%
114,Client E2E p50 (s),Multi Run (10x),1024,1,1.831,2.038,+0.206,+11.3%
115,Client E2E p50 (s),Multi Run (10x),1024,2,3.606,2.692,-0.914,-25.3%
116,Client E2E p50 (s),Multi Run (10x),1024,4,2.886,3.243,+0.357,+12.4%
117,Client E2E p50 (s),Multi Run (10x),1024,8,3.274,3.982,+0.709,+21.6%


### TEE Overhead Attribution per Scenario

In [27]:
DIMS = ['run_type', 'num_output_tokens', 'num_concurrent_requests']

df_tee_oh = df_all[df_all['label'] == 'Encrypted'].copy()
df_tee_oh = df_tee_oh.merge(
    df_all[df_all['label'] == 'Non-Encrypted'][DIMS + ['client_ttft_s_p50']].rename(
        columns={'client_ttft_s_p50': 'client_ttft_plain'}
    ),
    on=DIMS,
)

df_tee_oh['enc_time_s']      = df_tee_oh['total_encryption_time_ms_mean']  / 1000
df_tee_oh['dec_time_s']      = df_tee_oh['total_decryption_time_ms_mean']  / 1000
df_tee_oh['network_s']       = df_tee_oh['server_network_latency_ms_mean'] / 1000
df_tee_oh['tee_total_s']     = df_tee_oh['enc_time_s'] + df_tee_oh['dec_time_s'] + df_tee_oh['network_s']
df_tee_oh['ttft_delta_s']    = df_tee_oh['client_ttft_s_p50'] - df_tee_oh['client_ttft_plain']
df_tee_oh['network_pct']     = df_tee_oh['network_s']   / df_tee_oh['client_ttft_s_p50'] * 100
df_tee_oh['enc_pct']         = df_tee_oh['enc_time_s']  / df_tee_oh['client_ttft_s_p50'] * 100
df_tee_oh['dec_pct']         = df_tee_oh['dec_time_s']  / df_tee_oh['client_ttft_s_p50'] * 100
df_tee_oh['tee_total_pct']   = df_tee_oh['tee_total_s'] / df_tee_oh['client_ttft_s_p50'] * 100

DISPLAY = {
    'run_type':          'Run Type',
    'num_output_tokens': 'Output Tokens',
    'num_concurrent_requests': 'Concurrency',
    'client_ttft_plain': 'Plain TTFT (s)',
    'client_ttft_s_p50': 'Enc TTFT (s)',
    'ttft_delta_s':      'Delta (s)',
    'enc_time_s':        'Encryption (s)',
    'dec_time_s':        'Decryption (s)',
    'network_s':         'Network Hop (s)',
    'tee_total_s':       'TEE Total (s)',
    'tee_total_pct':     'TEE % of Enc TTFT',
    'network_pct':       'Network %',
    'enc_pct':           'Encrypt %',
    'dec_pct':           'Decrypt %',
}

df_tee_display = (
    df_tee_oh[list(DISPLAY.keys())]
    .rename(columns=DISPLAY)
    .sort_values(['Run Type', 'Output Tokens', 'Concurrency'])
)

pct_cols = [c for c in df_tee_display.columns if '%' in c]
s_cols   = [c for c in df_tee_display.columns if '(s)' in c]

df_tee_display.style \
    .background_gradient(subset=['TEE % of Enc TTFT'], cmap='YlOrRd') \
    .format({c: '{:.1f}%' for c in pct_cols}) \
    .format({c: '{:.4f}'  for c in s_cols}, na_rep='—')

,Run Type,Output Tokens,Concurrency,Plain TTFT (s),Enc TTFT (s),Delta (s),Encryption (s),Decryption (s),Network Hop (s),TEE Total (s),TEE % of Enc TTFT,Network %,Encrypt %,Decrypt %
0,Single Run (1x),100,1,0.4513,1.1692,0.7179,0.0046,0.0001,1.0602,1.0649,91.079550,90.673649,0.397537,0.008365
1,Single Run (1x),100,2,0.3772,0.7649,0.3877,0.0054,0.0008,0.8231,0.8294,108.430658,107.611139,0.712106,0.107413
2,Single Run (1x),100,4,0.7326,0.7800,0.0474,0.0043,0.0005,0.6726,0.6773,86.837397,86.230936,0.545821,0.060641
3,Single Run (1x),100,8,0.8543,0.9587,0.1044,0.0039,0.0004,0.8867,0.8910,92.938177,92.486617,0.408365,0.043194
4,Single Run (1x),100,16,0.9571,1.0157,0.0586,0.0039,0.0003,1.0618,1.0660,104.955459,104.533908,0.387477,0.034075
5,Single Run (1x),100,32,2.0984,1.7525,-0.3459,0.0039,0.0004,1.8571,1.8614,106.211863,105.970585,0.221261,0.020017
6,Single Run (1x),1024,1,0.1820,0.3549,0.1729,0.0049,0.0001,0.2517,0.2567,72.326374,70.926881,1.376444,0.023049
7,Single Run (1x),1024,2,0.2914,1.1630,0.8716,0.0045,0.0034,1.8682,1.8761,161.314832,160.638272,0.385795,0.290765
8,Single Run (1x),1024,4,0.3281,0.6831,0.3550,0.0043,0.0043,2.3957,2.4043,351.971644,350.710496,0.625955,0.635193
9,Single Run (1x),1024,8,2.2655,2.7299,0.4644,0.0042,0.0031,3.2319,3.2392,118.655980,118.389366,0.152793,0.113821
